In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
try:
    import orjson; loads = orjson.loads
except ImportError:
    loads = json.loads

WORK = "/kaggle/working"
DATA_DIR = "/kaggle/input/datasets/muttaqien19/dataset-suricata-logs"
files = sorted(p for p in glob.glob(f"{DATA_DIR}/**/eve.json.*", recursive=True) if os.path.isfile(p))
df = pd.read_parquet(glob.glob("/kaggle/input/**/ml_dataset.parquet", recursive=True)[0])
want = set(df["flow_id"].dropna().astype("int64").tolist())
print("Fichier :", files[0], "| flow_id recherchés :", len(want))

rows, sample, t0 = [], None, time.time()
with open(files[0], "rb") as f:
    for i, line in enumerate(f):
        if i and i % 5_000_000 == 0:
            print(f"{i:,} lignes | {len(rows):,} événements http gardés | {(time.time() - t0) / 60:.1f} min")
        if b'"event_type":"http"' not in line:
            continue
        try:
            e = loads(line)
        except Exception:
            continue
        if sample is None:
            sample = e
        fid = e.get("flow_id")
        if fid not in want:
            continue
        h = e.get("http", {})
        rows.append({"flow_id": fid, "timestamp": e.get("timestamp"),
                     "src_ip": e.get("src_ip"), "dest_ip": e.get("dest_ip"), "dest_port": e.get("dest_port"),
                     "hostname": h.get("hostname"), "url": (h.get("url") or "")[:500],
                     "user_agent": (h.get("http_user_agent") or "")[:300],
                     "method": h.get("http_method"), "status": h.get("status"), "length": h.get("length"),
                     "content_type": h.get("http_content_type"), "protocol": h.get("protocol")})

http = pd.DataFrame(rows)
http.to_parquet(f"{WORK}/http_events.parquet", index=False)
print(f"\nTerminé en {(time.time() - t0) / 60:.1f} min | événements http gardés : {len(http):,} "
      f"| flow_id distincts : {http['flow_id'].nunique():,}")
print("Champs http disponibles :", sorted(sample.get("http", {}).keys()))

# Couverture : quelle part des alertes du jeu ML a une requête HTTP lisible ?
df["has_http"] = df["flow_id"].astype("int64").isin(set(http["flow_id"]))
print("\nCouverture HTTP par direction :")
print(df.groupby("direction" if "direction" in df else df["dest_ip"].eq("192.168.33.8").map({True: "interne", False: "entrant"}))
        ["has_http"].mean().round(3).to_string())
print("\nCouverture HTTP par label :", df.groupby("label")["has_http"].mean().round(3).to_dict())
print("Couverture HTTP par port :", df.groupby("dest_port")["has_http"].mean().round(3).head(6).to_dict())
print("\nExemples :")
print(http[["method", "url", "user_agent", "status"]].head(8).to_string())

In [ ]:
# Direction (même définition que l'étape A) et même coupure temporelle que le notebook principal
is_int = lambda s: s.astype(str).str.startswith("192.168.")
df["direction"] = np.select([~is_int(df["src_ip"]) & is_int(df["dest_ip"]),
                             is_int(df["src_ip"]) & ~is_int(df["dest_ip"]),
                             is_int(df["src_ip"]) & is_int(df["dest_ip"])],
                            ["entrant", "sortant", "interne"], "externe")
t_cut = df.sort_values("ts", kind="mergesort")["ts"].iloc[int(len(df) * 0.8)]

http["ts_http"] = pd.to_datetime(http["timestamp"], utc=True, format="ISO8601").astype("datetime64[ns, UTC]")
http["flow_id"] = http["flow_id"].astype("int64")
d = df[df["direction"] == "entrant"].copy()
d["flow_id"] = d["flow_id"].astype("int64")
d["ts"] = d["ts"].astype("datetime64[ns, UTC]")

print("Requêtes HTTP par flux :",
      http.groupby("flow_id").size().describe(percentiles=[.5, .9, .99]).round(1).to_dict())

# Pour chaque alerte : la requête HTTP du même flux la plus proche dans le temps (±5 s)
m = pd.merge_asof(d.sort_values("ts"),
                  http.sort_values("ts_http")[["flow_id", "ts_http", "method", "url", "user_agent", "status"]],
                  left_on="ts", right_on="ts_http", by="flow_id",
                  direction="nearest", tolerance=pd.Timedelta(seconds=5))
m = m[m["url"].notna()].copy()
m["dt_s"] = (m["ts"] - m["ts_http"]).dt.total_seconds().abs()
m["text"] = (m["method"].fillna("") + " " + m["url"].fillna("") + " || " + m["user_agent"].fillna("")).str.strip()
m["split"] = np.where(m["ts"] < t_cut, "train", "test")

print(f"\nAlertes entrantes : {len(d):,} | avec requête associée : {len(m):,} ({len(m) / len(d):.1%})")
print("Écart alerte ↔ requête (s) :", m["dt_s"].describe(percentiles=[.5, .9, .99]).round(2).to_dict())
print("\nPar split :")
print(m.groupby("split")["label"].agg(lignes="size", prevalence="mean").round(3).to_string())
print("\nLongueur du texte (caractères) :", m["text"].str.len().describe(percentiles=[.5, .9, .99]).round(0).to_dict())

# Mémorisation possible : textes du test déjà vus à l'identique dans le train
tr_texts = set(m.loc[m["split"] == "train", "text"])
te = m[m["split"] == "test"]
print("\nTextes distincts :", m["text"].nunique(), "/", len(m))
print("Part du test dont le texte exact existe déjà dans le train :", round(te["text"].isin(tr_texts).mean(), 3))

print("\nExemples POSITIFS :")
print(m.loc[m["label"] == 1, ["sig", "text"]].sample(6, random_state=42).to_string(max_colwidth=90))
print("\nExemples NÉGATIFS :")
print(m.loc[m["label"] == 0, ["sig", "text"]].sample(6, random_state=42).to_string(max_colwidth=90))

cols = ["flow_id", "ts", "src_ip", "dest_port", "label", "sig", "method", "url", "user_agent", "status", "text", "split"]
m[cols].to_parquet(f"{WORK}/text_dataset.parquet", index=False)
print("\nSauvegardé : text_dataset.parquet")

In [ ]:
import re, time
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score

SEED = 42
tr = (m["split"] == "train").values
te = ~tr
y = m["label"].values
y_te = y[te]
tr_texts = set(m.loc[tr, "text"])
te_new = ~m.loc[te, "text"].isin(tr_texts).values      # textes du test jamais vus dans le train

def evals(p, mask=None):
    mask = np.ones(len(y_te), bool) if mask is None else mask
    yy, pp = y_te[mask], p[mask]
    pr = (pp >= 0.5).astype(int)
    return {"n": int(mask.sum()), "prevalence": round(float(yy.mean()), 3),
            "auc_pr": average_precision_score(yy, pp), "roc_auc": roc_auc_score(yy, pp),
            "precision": precision_score(yy, pr, zero_division=0), "recall": recall_score(yy, pr, zero_division=0)}

# --- Modèle texte classique : TF-IDF sur n-grammes de caractères + régression logistique ---
t0 = time.time()
vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=200_000,
                      sublinear_tf=True, lowercase=False, dtype=np.float32)
T_tr = vec.fit_transform(m.loc[tr, "text"])          # vocabulaire appris sur le TRAIN uniquement
T_te = vec.transform(m.loc[te, "text"])
lr_txt = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="liblinear").fit(T_tr, y[tr])
p_tfidf = lr_txt.predict_proba(T_te)[:, 1]
print(f"TF-IDF + LR : {T_tr.shape[1]:,} n-grammes | {time.time() - t0:.0f} s")

# --- Référence : LightGBM sur les variables de flux, MÊMES lignes (sous-ensemble HTTP) ---
host_feats = ["hist_flows", "hist_alerted", "hist_alert_rate", "is_new_host",
              "prev1h_flows", "prev1h_alerted", "prev1h_ports"]
cat = ["proto", "app_proto", "state", "reason"]
num = ["dest_port", "pkts_toserver", "pkts_toclient", "bytes_toserver", "bytes_toclient",
       "age", "bytes_ratio", "pkts_total", "hour", "dow", "ip_prev_alerts", "ip_gap_s"] + host_feats
Xf = pd.get_dummies(m[num + cat], columns=cat, dummy_na=True).astype("float32").fillna(0)
Xf.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in Xf.columns]
lgb_f = LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1).fit(Xf[tr], y[tr])
p_lgbf = lgb_f.predict_proba(Xf[te])[:, 1]

rows = {}
for name, p in [("LightGBM (flux)", p_lgbf), ("TF-IDF + LR (texte)", p_tfidf)]:
    rows[(name, "test complet")] = evals(p)
    rows[(name, "textes nouveaux")] = evals(p, te_new)
    rows[(name, "textes déjà vus")] = evals(p, ~te_new)
res = pd.DataFrame(rows).T
print("\nPart des textes nouveaux dans le test :", round(float(te_new.mean()), 3))
print(res.round(3).to_string())

# Rappel par famille (test) pour les deux modèles
fam = pd.DataFrame({"sig": m.loc[te, "sig"].values, "y": y_te,
                    "lgbm_flux": (p_lgbf >= 0.5).astype(int), "tfidf_texte": (p_tfidf >= 0.5).astype(int)})
pos = fam[fam["y"] == 1].groupby("sig").agg(n=("y", "size"), rappel_lgbm=("lgbm_flux", "mean"),
                                            rappel_texte=("tfidf_texte", "mean"))
print("\nRappel par famille positive (test, top 10) :")
print(pos.sort_values("n", ascending=False).head(10).round(2).to_string())
neg = fam[fam["y"] == 0].groupby("sig").agg(n=("y", "size"), fp_lgbm=("lgbm_flux", "mean"), fp_texte=("tfidf_texte", "mean"))
print("\nTaux de faux positifs par famille négative (test, top 6) :")
print(neg.sort_values("n", ascending=False).head(6).round(2).to_string())

In [ ]:
# Leave-one-family-out : on RETIRE une famille d'attaque de l'entraînement, puis on regarde
# si le modèle la détecte quand même dans le test (= capacité à reconnaître une attaque jamais vue).
sig_tr = m.loc[tr, "sig"].values
sig_te = m.loc[te, "sig"].values
fams = (pd.Series(sig_te[y_te == 1]).value_counts().head(8).index.tolist())

out = []
t0 = time.time()
for F in fams:
    keep = ~((sig_tr == F) & (y[tr] == 1))             # train sans aucun positif de la famille F
    idx_tr = np.where(tr)[0][keep]
    # Texte : vocabulaire ET modèle réappris sans F (strict)
    v = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=200_000,
                        sublinear_tf=True, lowercase=False, dtype=np.float32)
    A = v.fit_transform(m["text"].values[idx_tr])
    lr_F = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="liblinear").fit(A, y[idx_tr])
    pt = lr_F.predict_proba(v.transform(m.loc[te, "text"]))[:, 1]
    # Flux : LightGBM réappris sans F
    lg_F = LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1)
    lg_F.fit(Xf.iloc[idx_tr], y[idx_tr])
    pl = lg_F.predict_proba(Xf[te])[:, 1]

    isF = (sig_te == F) & (y_te == 1)                   # positifs de la famille retirée
    neg = (y_te == 0)
    sub = isF | neg                                     # famille F contre tous les négatifs du test
    for name, p in [("LightGBM (flux)", pl), ("TF-IDF + LR (texte)", pt)]:
        out.append({"famille retirée": F[:55], "modèle": name, "n_F": int(isF.sum()),
                    "rappel_F": float((p[isF] >= 0.5).mean()),
                    "auc_pr_F_vs_neg": average_precision_score(y_te[sub], p[sub]),
                    "taux_FP": float((p[neg] >= 0.5).mean())})
    print(f"{F[:55]:55s} fait ({time.time() - t0:.0f} s)")

lofo = pd.DataFrame(out)
print("\nRappel sur la famille JAMAIS vue à l'entraînement (seuil 0,5) :")
print(lofo.pivot(index="famille retirée", columns="modèle", values="rappel_F").round(2).to_string())
print("\nAUC-PR famille retirée contre négatifs (référence = part de F dans ce sous-ensemble) :")
print(lofo.pivot(index="famille retirée", columns="modèle", values="auc_pr_F_vs_neg").round(3).to_string())
print("\nMoyenne sur les 8 familles :")
print(lofo.groupby("modèle")[["rappel_F", "auc_pr_F_vs_neg", "taux_FP"]].mean().round(3).to_string())

In [ ]:
moz = m["user_agent"].fillna("").str.contains("Mozlila|Bulid", regex=True)
print("Alertes avec l'user-agent 'Mozlila/Bulid' :", int(moz.sum()), f"({moz.mean():.1%})")
print("Taux de positifs avec / sans :", round(m.loc[moz, "label"].mean(), 3), "/", round(m.loc[~moz, "label"].mean(), 3))
print("IP distinctes qui l'utilisent :", m.loc[moz, "src_ip"].nunique())
print("\nFamilles concernées :")
print(m.loc[moz, "sig"].value_counts().head(8).to_string())

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAXLEN = 300                                          # 99 % des textes font moins de 302 caractères

def encode(texts):
    """Chaque caractère → son octet (1..256), 0 = remplissage."""
    arr = np.zeros((len(texts), MAXLEN), dtype=np.int64)
    for i, t in enumerate(texts):
        b = t.encode("utf-8", "ignore")[:MAXLEN]
        arr[i, :len(b)] = np.frombuffer(b, dtype=np.uint8).astype(np.int64) + 1
    return arr

E_all = encode(m["text"].tolist())
idx_tr_all = np.where(tr)[0]
v_cut = int(len(idx_tr_all) * 0.8)                   # validation = 20 % les plus récents du train
i_fit, i_val, i_te = idx_tr_all[:v_cut], idx_tr_all[v_cut:], np.where(te)[0]
print("Ajustement / validation / test :", len(i_fit), len(i_val), len(i_te), "| device :", DEVICE)

class CharCNN(nn.Module):
    def __init__(self, emb=32, nf=128, ks=(3, 5, 7), drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(257, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, nf, k, padding=k // 2) for k in ks])
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(nf * len(ks), 1)

    def forward(self, x):
        e = self.emb(x).transpose(1, 2)                                   # (batch, emb, longueur)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)  # max-pooling global
        return self.out(self.drop(h)).squeeze(1)

def predict_cnn(model, idx, bs=2048):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            xb = torch.from_numpy(E_all[idx[i:i + bs]]).to(DEVICE)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)

def train_cnn(i_fit, i_val, max_epochs=10, patience=2, bs=256, lr=1e-3):
    torch.manual_seed(SEED)
    model = CharCNN().to(DEVICE)
    yf = y[i_fit].astype("float32")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((1 - yf.mean()) / yf.mean()), device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, wait, hist = -1.0, None, 0, []
    for ep in range(max_epochs):
        t0 = time.time(); model.train()
        perm = np.random.permutation(len(i_fit)); tot = 0.0
        for k in range(0, len(perm), bs):
            b = i_fit[perm[k:k + bs]]
            xb = torch.from_numpy(E_all[b]).to(DEVICE)
            yb = torch.from_numpy(y[b].astype("float32")).to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        ap_v = average_precision_score(y[i_val], predict_cnn(model, i_val))
        hist.append((ep + 1, tot / len(i_fit), ap_v))
        print(f"  époque {ep + 1:2d} | perte {tot / len(i_fit):.4f} | AUC-PR validation {ap_v:.4f} | {time.time() - t0:.0f} s")
        if ap_v > best + 1e-4:
            best, wait = ap_v, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best, pd.DataFrame(hist, columns=["epoch", "train_loss", "val_auc_pr"])

cnn, ap_val_cnn, hist_cnn = train_cnn(i_fit, i_val)
p_cnn = predict_cnn(cnn, i_te)

for sub_name, mask in [("test complet", None), ("textes nouveaux", te_new), ("textes déjà vus", ~te_new)]:
    rows[("Char-CNN (texte, DL)", sub_name)] = evals(p_cnn, mask)
res = pd.DataFrame(rows).T
print(f"\nAUC-PR validation Char-CNN : {ap_val_cnn:.3f}")
print(res.round(3).to_string())

fam["cnn_texte"] = (p_cnn >= 0.5).astype(int)
print("\nRappel par famille positive (test, top 10) :")
print(fam[fam["y"] == 1].groupby("sig").agg(n=("y", "size"), lgbm=("lgbm_flux", "mean"),
      tfidf=("tfidf_texte", "mean"), cnn=("cnn_texte", "mean")).sort_values("n", ascending=False).head(10).round(2).to_string())

torch.save({"state_dict": {k: v.cpu() for k, v in cnn.state_dict().items()}, "maxlen": MAXLEN,
            "config": {"emb": 32, "nf": 128, "ks": (3, 5, 7), "drop": 0.3}},
           f"{WORK}/model_charcnn.pt", _use_new_zipfile_serialization=False)
print("Modèle sauvegardé : model_charcnn.pt")

In [ ]:
!pip install -q mlflow
import shutil, mlflow

# 1) Prédictions de VALIDATION (modèles entraînés sur la partie ajustement seulement) pour choisir le poids
v_fit = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=200_000,
                        sublinear_tf=True, lowercase=False, dtype=np.float32)
A_fit = v_fit.fit_transform(m["text"].values[i_fit])
pv_t = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="liblinear") \
           .fit(A_fit, y[i_fit]).predict_proba(v_fit.transform(m["text"].values[i_val]))[:, 1]
pv_l = LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1) \
           .fit(Xf.iloc[i_fit], y[i_fit]).predict_proba(Xf.iloc[i_val])[:, 1]
pv_c = predict_cnn(cnn, i_val)

ws = np.round(np.linspace(0, 1, 11), 1)
def best_w(p_flux, p_txt):
    aps = [average_precision_score(y[i_val], w * p_flux + (1 - w) * p_txt) for w in ws]
    return float(ws[int(np.argmax(aps))]), max(aps)

w_t, ap_t = best_w(pv_l, pv_t)
w_c, ap_c = best_w(pv_l, pv_c)
print(f"Poids du flux choisi sur la validation : avec TF-IDF = {w_t} (AUC-PR val {ap_t:.3f}) | "
      f"avec Char-CNN = {w_c} (AUC-PR val {ap_c:.3f})")
print(f"Références validation : LightGBM seul {average_precision_score(y[i_val], pv_l):.3f} | "
      f"TF-IDF seul {average_precision_score(y[i_val], pv_t):.3f} | CNN seul {average_precision_score(y[i_val], pv_c):.3f}")

# 2) Application au TEST (poids figés)
p_fus_t = w_t * p_lgbf + (1 - w_t) * p_tfidf
p_fus_c = w_c * p_lgbf + (1 - w_c) * p_cnn
for name, p in [(f"Fusion flux + TF-IDF (w={w_t})", p_fus_t), (f"Fusion flux + CNN (w={w_c})", p_fus_c)]:
    for sub_name, mask in [("test complet", None), ("textes nouveaux", te_new), ("textes déjà vus", ~te_new)]:
        rows[(name, sub_name)] = evals(p, mask)
res = pd.DataFrame(rows).T
print("\n", res.round(3).to_string())

fam["fusion_tfidf"] = (p_fus_t >= 0.5).astype(int)
print("\nRappel par famille (test) :")
print(fam[fam["y"] == 1].groupby("sig").agg(n=("y", "size"), lgbm=("lgbm_flux", "mean"), tfidf=("tfidf_texte", "mean"),
      cnn=("cnn_texte", "mean"), fusion=("fusion_tfidf", "mean")).sort_values("n", ascending=False).head(10).round(2).to_string())

# 3) MLflow : un run par modèle (périmètre : alertes entrantes avec requête HTTP)
dbs = sorted(glob.glob("/kaggle/input/**/mlflow.db", recursive=True), key=lambda p: "deep" not in p)
if dbs and not os.path.exists(f"{WORK}/mlflow.db"):
    shutil.copy(dbs[0], f"{WORK}/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{WORK}/mlflow.db")
if mlflow.get_experiment_by_name("suricata-detection") is None:
    mlflow.create_experiment("suricata-detection", artifact_location=f"{WORK}/mlartifacts")
mlflow.set_experiment("suricata-detection")
lofo_mean = lofo.groupby("modèle")[["rappel_F", "auc_pr_F_vs_neg"]].mean()
for run, model_name in [("http_lgbm_flux", "LightGBM (flux)"), ("http_tfidf_lr", "TF-IDF + LR (texte)"),
                        ("http_charcnn", "Char-CNN (texte, DL)"), ("http_fusion_tfidf", f"Fusion flux + TF-IDF (w={w_t})"),
                        ("http_fusion_cnn", f"Fusion flux + CNN (w={w_c})")]:
    with mlflow.start_run(run_name=run):
        mlflow.log_params({"perimetre": "entrant + requête HTTP", "n_test": int(te.sum()), "modele": model_name})
        mlflow.log_metrics({f"{k}_complet": float(v) for k, v in res.loc[(model_name, "test complet")].items() if k != "n"} |
                           {f"{k}_nouveaux": float(v) for k, v in res.loc[(model_name, "textes nouveaux")].items() if k != "n"})
        if model_name in lofo_mean.index:
            mlflow.log_metrics({"lofo_rappel_moyen": float(lofo_mean.loc[model_name, "rappel_F"]),
                                "lofo_auc_pr_moyen": float(lofo_mean.loc[model_name, "auc_pr_F_vs_neg"])})
        if run == "http_charcnn":
            mlflow.log_artifact(f"{WORK}/model_charcnn.pt")
res.round(4).to_csv(f"{WORK}/resultats_http.csv")
print("\nRuns MLflow enregistrés | base :", dbs[0] if dbs else "nouvelle")

In [ ]:
out_cnn = []
t0 = time.time()
sig_all = m["sig"].values
for F in fams:
    drop_F = (sig_all == F) & (y == 1)
    i_fit_F, i_val_F = i_fit[~drop_F[i_fit]], i_val[~drop_F[i_val]]     # F absente de l'entraînement ET de la validation
    cnn_F, _, _ = train_cnn(i_fit_F, i_val_F, max_epochs=1, patience=1)
    pc = predict_cnn(cnn_F, i_te)
    isF = (sig_te == F) & (y_te == 1); neg = (y_te == 0); sub = isF | neg
    out_cnn.append({"famille retirée": F[:55], "modèle": "Char-CNN (texte, DL)", "n_F": int(isF.sum()),
                    "rappel_F": float((pc[isF] >= 0.5).mean()),
                    "auc_pr_F_vs_neg": average_precision_score(y_te[sub], pc[sub]),
                    "taux_FP": float((pc[neg] >= 0.5).mean())})
    print(f"{F[:55]:55s} fait ({(time.time() - t0) / 60:.1f} min)")

lofo_all = pd.concat([lofo, pd.DataFrame(out_cnn)], ignore_index=True)
print("\nRappel sur la famille jamais vue :")
print(lofo_all.pivot(index="famille retirée", columns="modèle", values="rappel_F").round(2).to_string())
print("\nMoyenne sur les 8 familles :")
print(lofo_all.groupby("modèle")[["rappel_F", "auc_pr_F_vs_neg", "taux_FP"]].mean().round(3).to_string())
lofo_all.to_csv(f"{WORK}/lofo_http.csv", index=False)

In [ ]:
url_only = (m["method"].fillna("") + " " + m["url"].fillna("")).str.strip()
ua_only = m["user_agent"].fillna("")

def tfidf_eval(texts):
    v = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=200_000,
                        sublinear_tf=True, lowercase=False, dtype=np.float32)
    A = v.fit_transform(texts.values[tr])
    lr_ = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="liblinear").fit(A, y[tr])
    return lr_.predict_proba(v.transform(texts.values[te]))[:, 1]

p_url, p_ua = tfidf_eval(url_only), tfidf_eval(ua_only)
moz_te = moz.values[te]

rows2 = {}
for name, p in [("LightGBM (flux)", p_lgbf), ("TF-IDF URL + UA", p_tfidf),
                ("TF-IDF URL seule", p_url), ("TF-IDF user-agent seul", p_ua), ("Char-CNN URL + UA", p_cnn)]:
    rows2[(name, "test complet")] = evals(p)
    rows2[(name, "hors Mozlila")] = evals(p, ~moz_te)
    rows2[(name, "hors Mozlila, textes nouveaux")] = evals(p, ~moz_te & te_new)
print("Alertes Mozlila dans le test :", int(moz_te.sum()), "| positifs du test qui sont Mozlila :",
      round(float(moz_te[y_te == 1].mean()), 3))
print(pd.DataFrame(rows2).T.round(3).to_string())

In [ ]:
# Système hybride : modèle TEXTE quand la requête HTTP existe, sinon LightGBM sur les FLUX
d_all = df[df["direction"] == "entrant"].copy()
d_all["flow_id"] = d_all["flow_id"].astype("int64")
assert d_all["flow_id"].is_unique, "flow_id non unique : la correspondance texte ↔ alerte serait ambiguë"
Xd = pd.get_dummies(d_all[num + cat], columns=cat, dummy_na=True).astype("float32").fillna(0)
Xd.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in Xd.columns]
yd = d_all["label"].values
trd = (d_all["ts"] < t_cut).values
ted = ~trd

lg_d = LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1).fit(Xd[trd], yd[trd])
p_flux = lg_d.predict_proba(Xd[ted])[:, 1]

txt = pd.Series(p_tfidf, index=m.loc[te, "flow_id"].astype("int64").values).groupby(level=0).first()
fid_te = d_all.loc[ted, "flow_id"].values
has_txt = pd.Index(txt.index).get_indexer(fid_te) >= 0
p_hyb = p_flux.copy()
p_hyb[has_txt] = txt.loc[fid_te[has_txt]].values
y_d = yd[ted]

def ev(yy, pp):
    pr = (pp >= 0.5).astype(int)
    return {"n": len(yy), "prevalence": round(float(yy.mean()), 3), "auc_pr": average_precision_score(yy, pp),
            "roc_auc": roc_auc_score(yy, pp), "precision": precision_score(yy, pr, zero_division=0),
            "recall": recall_score(yy, pr, zero_division=0)}

tab = {}
for sub_name, mk in [("tout le test entrant", np.ones(len(y_d), bool)), ("avec HTTP", has_txt), ("sans HTTP", ~has_txt)]:
    tab[("LightGBM flux seul", sub_name)] = ev(y_d[mk], p_flux[mk])
    tab[("Hybride texte / flux", sub_name)] = ev(y_d[mk], p_hyb[mk])
print(f"Test entrant : {len(y_d):,} alertes | avec requête HTTP : {has_txt.mean():.1%}")
print(pd.DataFrame(tab).T.round(3).to_string())

# IC 95 % bootstrap par IP (apparié) sur tout le test entrant
codes, _ = pd.factorize(d_all.loc[ted, "src_ip"])
groups = pd.Series(np.arange(len(y_d))).groupby(codes).apply(np.array).values
rng, bh, bd = np.random.RandomState(SEED), [], []
for _ in range(300):
    ii = np.concatenate(groups[rng.randint(0, len(groups), len(groups))])
    if y_d[ii].sum():
        a_h, a_f = average_precision_score(y_d[ii], p_hyb[ii]), average_precision_score(y_d[ii], p_flux[ii])
        bh.append(a_h); bd.append(a_h - a_f)
ci_h, ci_d = np.percentile(bh, [2.5, 97.5]), np.percentile(bd, [2.5, 97.5])
print(f"\nHybride AUC-PR = {average_precision_score(y_d, p_hyb):.3f} IC 95 % {ci_h.round(3)}")
print(f"Gain hybride − flux = {np.mean(bd):.3f} IC 95 % {ci_d.round(3)}")

# Export pour le dashboard
pred_h = d_all.loc[ted, ["flow_id", "ts", "src_ip", "dest_port", "label", "sig"]].copy()
pred_h["p_flux"] = p_flux
pred_h["p_texte"] = np.where(has_txt, p_hyb, np.nan)
pred_h["p_final"] = p_hyb
pred_h["source"] = np.where(has_txt, "texte_http", "flux")
pred_h["risk"] = (100 * p_hyb).round(1)
pred_h.to_parquet(f"{WORK}/predictions_hybride.parquet", index=False)
print("Sauvegardé : predictions_hybride.parquet", pred_h.shape)

with mlflow.start_run(run_name="systeme_hybride_texte_flux"):
    mlflow.log_params({"perimetre": "trafic entrant", "regle": "TF-IDF texte si requête HTTP, sinon LightGBM flux"})
    mlflow.log_metrics({f"{k}": float(v) for k, v in tab[("Hybride texte / flux", "tout le test entrant")].items() if k != "n"} |
                       {"auc_pr_ci_low": float(ci_h[0]), "auc_pr_ci_high": float(ci_h[1]),
                        "gain_vs_flux": float(np.mean(bd)), "gain_ci_low": float(ci_d[0]), "gain_ci_high": float(ci_d[1])})
    for f in ["predictions_hybride.parquet", "resultats_http.csv", "lofo_http.csv"]:
        if os.path.exists(f"{WORK}/{f}"):
            mlflow.log_artifact(f"{WORK}/{f}")